# Kaggriculture V43R — Hard-Regime Atlas

This runs in parallel with V41.2. It identifies observable game regimes where exact V32 repeatedly loses to the Melon / Ranker / Adaptive frontier family.

It records V32 state at turns 144, 216, 288, and 360 across 64 fresh seeds, then fits a tiny interpretable regime classifier and ranks hard-vs-easy feature shifts.

## Kaggle settings
- Accelerator: None / CPU
- Internet: ON
- Required: exact V32
- Strongly recommended: Moon/Melon, Ranker, Adaptive
- Also useful: Soil, Strict, WEED-Slip


In [ ]:
from pathlib import Path
import os,sys,subprocess,shutil,json
import pandas as pd
PIN='2532bc376cfe5c271e7c74302d0df3a219f38f83'
INPUT=Path('/kaggle/input')
REPO=Path('/kaggle/working/kaggriculture_v43_atlas_source')
WORK=Path('/kaggle/working/v43_regime_atlas')
print('='*72)
print('V43_HARD_REGIME_ATLAS_BOOTSTRAP_OK')
print('='*72)
print('python:',sys.version)
print('cpu:',os.cpu_count())
v32=list(INPUT.rglob('SUBMIT_V32_RUNTIME_VERIFIED.tar.gz')) or list(INPUT.rglob('SUBMIT_V32_PREMIUM_FRONT_SINGLEFILE.tar.gz'))
assert v32,'Missing exact V32 archive'
print('V32:',v32[0])


In [ ]:
patterns={'melon':'kaggriculture-frontier-the-moon-counts-melons','ranker':'kaggriculture-rank-your-agent','adaptive':'adaptive-farming-strategy-for-kaggriculture','soil':'kaggriculture-frontier-the-soil-remembers-rain','strict':'25-27-strict-future-v27-midgame-meta-reset','weed_slip':'weed-slip'}
rows=[]
for k,p in patterns.items():
    hits=[x for x in INPUT.rglob('*') if p in str(x).lower() and x.name in {'main.py','submission.tar.gz'}]
    rows.append({'family':k,'found':bool(hits),'example':str(hits[0]) if hits else ''})
display(pd.DataFrame(rows))


In [ ]:
if REPO.exists():shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','https://github.com/sidhulyalkar/kaggriculture.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach',PIN],check=True,capture_output=True)
head=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert head==PIN,(head,PIN)
SCRIPT=REPO/'scripts'/'v43_hard_regime_atlas.py'
assert SCRIPT.exists(),SCRIPT
compile(SCRIPT.read_text(),str(SCRIPT),'exec')
print('V43R SOURCE READY:',head)


In [ ]:
if WORK.exists():shutil.rmtree(WORK)
cmd=[sys.executable,str(SCRIPT),'--input-root',str(INPUT),'--repo',str(REPO),'--work',str(WORK),'--workers',str(min(4,os.cpu_count() or 2))]
print('RUN:',' '.join(cmd))
proc=subprocess.run(cmd)
if proc.returncode:
    print('V43R PIPELINE FAILED:',proc.returncode)
    p=WORK/'V43_REGIME_GAMES.csv'
    if p.exists():
        df=pd.read_csv(p);display(df.head(50))
        if 'ok' in df.columns:display(df[df.ok!=True].head(30))
    raise RuntimeError('V43R stopped; diagnostics above')
print('V43R PIPELINE OK')


In [ ]:
decision=json.loads((WORK/'V43_REGIME_DECISION.json').read_text())
display(pd.DataFrame([{'hard_seeds':decision.get('hard_seeds'),'easy_seeds':decision.get('easy_seeds'),'frontier_family':', '.join(decision.get('frontier_family',[])),'cv_auc':decision.get('best_tree_cv',{}).get('cv_auc'),'cv_balanced_accuracy':decision.get('best_tree_cv',{}).get('cv_balanced_accuracy')}]))
print('\n=== INTERPRETABLE REGIME RULES ===')
print((WORK/'V43_REGIME_RULES.txt').read_text())
display(pd.read_csv(WORK/'V43_REGIME_DIFFERENCES.csv').head(30))


In [ ]:
for fn in ['V43_REGIME_DECISION.json','V43_REGIME_GAMES.csv','V43_REGIME_FEATURES.csv','V43_REGIME_MODEL_CV.csv','V43_REGIME_RULES.txt','V43_REGIME_DIFFERENCES.csv']:
    p=WORK/fn
    if p.exists():shutil.copy2(p,Path('/kaggle/working')/fn)
print('V43 REGIME ATLAS COMPLETE')
print('Send back: V43_REGIME_DECISION.json, V43_REGIME_RULES.txt, V43_REGIME_DIFFERENCES.csv')
